# 00 · Pre-config — RetailHub trainer setup

Trainer-only notebook. It prepares isolated course environments at catalog level:

1. resolve participants from `TRAINING_GROUP` or `TRAINING_USERS`,
2. create one Unity Catalog catalog per participant: `retailhub_<user>`,
3. map trainer accounts to `retailhub_trainer`,
4. create `default`, `bronze`, `silver` and `gold` schemas plus the `datasets` volume,
5. run `data/generate_training_dataset.ipynb` once per catalog using `TARGET_CATALOG`.

Before running, confirm `STORAGE_LOCATION` points to a valid Unity Catalog managed storage location for this workspace.

In [0]:
import re
import requests

# Prefer the same group-based pattern as the Data Engineer Associate setup.
# Set TRAINING_USERS manually when the group is not available yet.
TRAINING_GROUP = "alt_trn_gr"
TRAINING_USERS = []  # Example: ["jan.kowalski@example.com", "anna.nowak@example.com"]

# Dataset generation is intentionally delegated to the real dataset notebook.
RUN_DATASET_GENERATION = True
DATASET_NOTEBOOK_PATH = "../data/generate_training_dataset"
DATASET_TIMEOUT_SECONDS = 0

CATALOG_PREFIX = "retailhub"

# Managed location for catalogs. Example Azure:
# "abfss://container@account.dfs.core.windows.net/path"
STORAGE_LOCATION = "abfss://training@dbxaltextstorage2.dfs.core.windows.net/"

DEFAULT_SCHEMA = "default"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
SCHEMAS = [DEFAULT_SCHEMA, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]
VOLUME_NAME = "datasets"

def get_group_members(group_name):
    context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host = context.apiUrl().get()
    token = context.apiToken().get()
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    groups_url = f"{host}/api/2.0/preview/scim/v2/Groups"
    response = requests.get(groups_url, headers=headers, params={"filter": f'displayName eq "{group_name}"'})
    response.raise_for_status()
    groups = response.json().get("Resources", [])
    if not groups:
        raise ValueError(f"Group '{group_name}' not found")

    group_url = f"{host}/api/2.0/preview/scim/v2/Groups/{groups[0]['id']}"
    response = requests.get(group_url, headers=headers)
    response.raise_for_status()

    users = []
    for member in response.json().get("members", []):
        if not member.get("$ref", "").startswith("Users/"):
            continue
        user_url = f"{host}/api/2.0/preview/scim/v2/Users/{member['value']}"
        user_response = requests.get(user_url, headers=headers)
        user_response.raise_for_status()
        emails = user_response.json().get("emails", [])
        if emails and emails[0].get("value"):
            users.append(emails[0]["value"])
    return users

# Kept in sync manually with resolve_user_slug() in setup/00_setup.ipynb —
# this notebook cannot %run that one, since it provisions the catalog that
# 00_setup expects to already exist.
def resolve_user_slug(user_name):
    identity = user_name.lower()
    local_part = identity.split("@")[0]
    if "trainer" in identity or "trener" in identity or local_part == "krzysztof.burejza":
        return "trainer"
    user_slug = re.sub(r"[^a-zA-Z0-9]", "_", local_part).lower()
    return re.sub(r"_+", "_", user_slug).strip("_") or "user"

def catalog_for_user(user_name):
    return f"{CATALOG_PREFIX}_{resolve_user_slug(user_name)}"

def get_training_users():
    explicit_users = sorted({u.strip() for u in TRAINING_USERS if u.strip()})
    if explicit_users:
        return explicit_users
    if not TRAINING_GROUP:
        return [spark.sql("SELECT current_user()").first()[0]]
    return sorted(set(get_group_members(TRAINING_GROUP)))

trainer_user = spark.sql("SELECT current_user()").first()[0]
training_users = get_training_users()

catalog_users = {}
for user in training_users:
    catalog_users.setdefault(catalog_for_user(user), set()).add(user)

# Ensure the trainer catalog exists when the trainer identity maps to retailhub_trainer.
trainer_catalog = catalog_for_user(trainer_user)
if trainer_catalog == f"{CATALOG_PREFIX}_trainer":
    catalog_users.setdefault(trainer_catalog, set()).add(trainer_user)

print("Trainer user:", trainer_user)
print("Training group:", TRAINING_GROUP or "(not used)")
print("Storage location:", STORAGE_LOCATION or "(metastore default)")
print("Dataset generation:", RUN_DATASET_GENERATION)
print("Catalog isolation level: one catalog per participant")
print()

for catalog_name, users in sorted(catalog_users.items()):
    print(f"{catalog_name}: {', '.join(sorted(users))}")

In [0]:
# Drop all existing retailhub_* catalogs before recreating them from scratch.
existing_catalogs = [
    r[0] for r in spark.sql("SHOW CATALOGS").collect()
    if r[0].startswith(f"{CATALOG_PREFIX}_")
]

if existing_catalogs:
    print(f"Dropping {len(existing_catalogs)} existing catalog(s):")
    for cat in sorted(existing_catalogs):
        spark.sql(f"DROP CATALOG IF EXISTS {cat} CASCADE")
        print(f"  Dropped: {cat}")
else:
    print("No existing retailhub_* catalogs found — nothing to drop.")
print()

In [0]:
def create_catalog_environment(catalog_name, users, storage_location):
    result = {"catalog": catalog_name, "users": sorted(users), "schemas": [], "volume": False, "grants": []}
    managed_location = f"{storage_location.rstrip('/')}/{catalog_name}" if storage_location else ""

    if managed_location:
        spark.sql(f"""
            CREATE CATALOG IF NOT EXISTS {catalog_name}
            MANAGED LOCATION '{managed_location}'
        """)
    else:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
    result["managed_location"] = managed_location or "metastore default"

    for schema in SCHEMAS:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema}")
        result["schemas"].append(schema)

    spark.sql(f"""
        CREATE VOLUME IF NOT EXISTS {catalog_name}.{DEFAULT_SCHEMA}.{VOLUME_NAME}
        COMMENT 'Training datasets for RetailHub Medicover course'
    """)
    result["volume"] = True

    principals = set(users)
    principals.add(trainer_user)
    for principal in sorted(principals):
        spark.sql(f"GRANT ALL PRIVILEGES ON CATALOG {catalog_name} TO `{principal}`")
        result["grants"].append(principal)

    return result

creation_results = {}
creation_errors = {}

for catalog_name, users in sorted(catalog_users.items()):
    print(f"Creating environment: {catalog_name}")
    try:
        creation_results[catalog_name] = create_catalog_environment(catalog_name, users, STORAGE_LOCATION)
        print(f"  schemas: {', '.join(creation_results[catalog_name]['schemas'])}")
        print(f"  volume : {creation_results[catalog_name]['volume']}")
        print(f"  grants : {', '.join(creation_results[catalog_name]['grants'])}")
    except Exception as e:
        creation_errors[catalog_name] = str(e)
        print(f"  ERROR: {e}")
    print()

if creation_errors:
    raise Exception(f"Catalog provisioning failed for: {sorted(creation_errors)}")

print(f"[OK] Provisioned {len(creation_results)} isolated catalog environments")

In [0]:
dataset_results = {}
dataset_errors = {}

if RUN_DATASET_GENERATION:
    for catalog_name in sorted(creation_results):
        print(f"Generating dataset for {catalog_name}...")
        try:
            run_result = dbutils.notebook.run(
                DATASET_NOTEBOOK_PATH,
                DATASET_TIMEOUT_SECONDS,
                {"TARGET_CATALOG": catalog_name},
            )
            dataset_results[catalog_name] = run_result
            print(f"  OK: {run_result}")
        except Exception as e:
            dataset_errors[catalog_name] = str(e)
            print(f"  ERROR: {e}")
        print()
else:
    print("Dataset generation skipped. Participants must run data/generate_training_dataset.ipynb in their own catalog.")

if dataset_errors:
    raise Exception(f"Dataset generation failed for: {sorted(dataset_errors)}")

print(f"[OK] Dataset generation completed for {len(dataset_results)} catalogs")

## Next step

Participants should start with `setup/00_setup.ipynb`. It resolves their own `retailhub_<user>` catalog.

If a participant catalog needs to be rebuilt, run `data/generate_training_dataset.ipynb` with `TARGET_CATALOG` set to that catalog, or rerun this pre-config notebook.